In [ ]:
#importlibraries

#libraries to extract frame
import os
import pandas as pd
import cv2
import numpy as np
import shutil

#libraries to undergo differencing
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from torchvision.utils import save_image
from sklearn.decomposition import PCA

#create a function that transforms images to tensors
convert_tensor = transforms.ToTensor()
convert_image = transforms.ToPILImage()

#set directories
#Imgae_to_video_locator_csv
#image_to_vid_file = '../../data/Deepfish/location_data/location_dat.csv'
#image_to_vid_file = '../../data/Kakadu_fish/location_data/location_dat.csv'
image_to_vid_file = '../../data/Tassy_BRUV/location_data/location_dat.csv'
#training data location
#folder with images, labels and yaml file are assumed to be within training data folder
#training_dat_loc = '../../data/Deepfish/training_data/'
#training_dat_loc = '../../data/Kakadu_fish/training_data/'
training_dat_loc = '../../data/Tassy_BRUV/training_data/'

#image size (if resizing needed for surrounding images)
#image_size = [1920,1080]
image_size = False
#if you do not want to resize surrounding images replace with False

#pixel location of time_stamp in video if wanted to be removed
#this blacks out top left corner of video to 635 pixels across and 70 down
#if you do not want to remove timestamp replace with False
#time_stamp_loc = [0,70,0,635] 
time_stamp_loc = False 

#whether to read in center frame from surrounding video/images
center_image_surround = True

#directory where differenced images will be saved
#diff_folder = '../../data/Deepfish/augmented_images/'
diff_folder = '../../data/Tassy_BRUV/augmented_images/'

#differenced image folder name
diff_name = 'test_zero_alt'

#differenced parameters
diff = 1
alpha = 10
method = "range"
red_layer = True
do_pca = True
normalise =False
log_im=True

In [ ]:
#images
image_loc = training_dat_loc + 'images/'

#import image location data
val_frame_dat = pd.read_csv(image_to_vid_file)

In [ ]:
val_frame_dat = pd.read_csv(image_to_vid_file)

In [1]:
do_pca ="all"
alpha=5
pca_lead_in = 5
method = "naive"

#im_num_list = list(range(0,val_frame_dat.shape[0]))
im_num_list = list(range(0,30))

for im_num in im_num_list:
    #read in center image
    if(center_image_surround==True):
            #read in video
        video = val_frame_dat['vid_location'][im_num]
        vidcap = cv2.VideoCapture(video)
        frame3 = val_frame_dat['frame'][im_num]
        vidcap.set(1, frame3)
        success,img_center = vidcap.read()
    else:
        img_center = Image.open(image_loc + val_frame_dat['split'][im_num] + "/" + val_frame_dat['image_name'][im_num])

    #convert images to tensors
    img_center_tens = convert_tensor(img_center)

    #fix video image before differencing
    if(center_image_surround==True):
        img_center_tens = torch.stack((img_center_tens[2,:,:],img_center_tens[1,:,:],img_center_tens[0,:,:]),dim=0)
    
    
    im_depth, im_height, im_width,  = img_center_tens.shape    
    # Reshape the image tensor into a 2D matrix
    im_matrix = np.reshape(convert_image(img_center_tens), (im_height * im_width, im_depth))
    
    if(im_num==0):
        #start off with the first image matrix
        im_matrix_all = im_matrix
    else:
        im_matrix_all = np.concatenate((im_matrix_all,im_matrix))

pca = PCA(n_components=2,whiten=True)
pca.fit(im_matrix_all)
            


NameError: name 'center_image_surround' is not defined

In [ ]:
iterate = True

#im_num_list = list(range(0,val_frame_dat.shape[0]))
im_num_list = list(range(0,2))

pca = PCA(n_components=2,whiten=True)

if(iterate==True):
    expl_var = []


for im_num in im_num_list:
    #read in center image
    if(center_image_surround==True):
            #read in video
        video = val_frame_dat['vid_location'][im_num]
        vidcap = cv2.VideoCapture(video)
        frame3 = val_frame_dat['frame'][im_num]
        vidcap.set(1, frame3)
        success,img_center = vidcap.read()
    else:
        img_center = Image.open(image_loc + val_frame_dat['split'][im_num] + "/" + val_frame_dat['image_name'][im_num])

    #convert images to tensors
    img_center_tens = convert_tensor(img_center)

    #fix video image before differencing
    if(center_image_surround==True):
        img_center_tens = torch.stack((img_center_tens[2,:,:],img_center_tens[1,:,:],img_center_tens[0,:,:]),dim=0)
    
    
    im_depth, im_height, im_width,  = img_center_tens.shape    
    # Reshape the image tensor into a 2D matrix
    im_matrix = np.reshape(convert_image(img_center_tens), (im_height * im_width, im_depth))
    
    
    
    if(iterate==True):
        pca.fit(im_matrix)
        expl_var.append([pca.explained_variance_ratio_[0],pca.explained_variance_ratio_[1]])
    else:    
        if(im_num==im_num_list[0]):
            #start off with the first image matrix
            im_matrix_all = im_matrix
        else:
            im_matrix_all = np.concatenate((im_matrix_all,im_matrix))

if(iterate==False):
    pca.fit(im_matrix_all)

In [ ]:
expl_var

In [ ]:
pca.explained_variance_ratio_